# Reconstruction diagnostics

Compare Giano, the two BiLSTM recipes and interpolation on the corrected validation benchmark: six variables, five paired training/mask seeds and thirteen missingness cases.

**Input:** the completed artifacts in `artifacts/evaluation/corrected_v2_release/` and the saved training configuration.
Change `SELECTED_VARIABLE` below to inspect one variable without mixing units.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from notebooks.release_artifacts import load_benchmarks  # noqa: E402

IMPUTED_ROOT = PROJECT_ROOT / "artifacts/imputed/release_giano"
SELECTED_VARIABLE = "temperature"  # keep units and variable-specific failures separate
IMPUTED_ROOT

## Same-mask comparison

MAE is the mean absolute error on artificially hidden observations; lower is better. The table retains all cases and seeds.
The chart shows each model's percentage improvement over same-mask interpolation, averaged across the five seeds: above zero is better, below zero is worse.


In [ ]:
rows, source_artifacts = load_benchmarks(PROJECT_ROOT)
results = pd.DataFrame(rows)
display({"split": "val", "seeds": [42, 43, 44, 45, 46], "sources": source_artifacts})
results.head()

In [ ]:
if not results.empty:
    results = results.copy()
    results["model_label"] = results["model"] + "/" + results["model_variant"]

    def mask_label(row):
        if row["mask_type"] == "point":
            return f"point {row['mask_parameter']:.0%}"
        if row["mask_type"] == "empirical":
            return f"empirical train q{row['mask_parameter']:.0%}"
        return f"{row['mask_type']} {int(row['mask_parameter'])}h"

    results["mask"] = results.apply(mask_label, axis=1)
    display(
        results[
            [
                "variable",
                "model_label",
                "seed",
                "mask",
                "mae",
                "rmse",
                "n_hidden",
            ]
        ]
    )

In [ ]:
if not results.empty:
    keys = ["variable", "seed", "mask"]
    baseline = (
        results[results["model"].eq("interpolation")][keys + ["mae"]]
        .drop_duplicates(keys)
        .rename(columns={"mae": "interpolation_mae"})
    )
    comparisons = results[~results["model"].eq("interpolation")].merge(
        baseline, on=keys, validate="many_to_one"
    )
    comparisons["mae_improvement_pct"] = (
        (comparisons["interpolation_mae"] - comparisons["mae"])
        / comparisons["interpolation_mae"].replace(0, np.nan)
        * 100
    )
    pivot = comparisons.query("variable == @SELECTED_VARIABLE").pivot_table(
        index="mask",
        columns="model_label",
        values="mae_improvement_pct",
        aggfunc="mean",
    )
    ax = pivot.plot(kind="bar", figsize=(14, 6), width=0.85)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_ylabel("MAE improvement over interpolation (%)")
    ax.set_title(f"{SELECTED_VARIABLE}: validation, mean over five paired seeds")
    ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()

## Optional: natural-gap reconstruction

This section reads separately generated inference NetCDF files from `IMPUTED_ROOT`. If none are present, it is skipped.
Select a file to inspect observations, filled points and their method labels. Natural gaps have no known ground truth, so this plot cannot establish accuracy.


In [ ]:
imputed_files = sorted(IMPUTED_ROOT.glob("*/*_imputed.nc"))
SELECTED_FILE = imputed_files[0] if imputed_files else None
if SELECTED_FILE is None:
    print("Optional natural-gap plot skipped: no inference files in", IMPUTED_ROOT)
SELECTED_FILE

In [ ]:
reconstruction = None
if SELECTED_FILE is not None:
    with xr.open_dataset(SELECTED_FILE, engine="h5netcdf") as source:
        reconstruction = source.load()
    summary = {
        "file": SELECTED_FILE.name,
        "points": reconstruction.sizes["time"],
        "imputed": int(reconstruction["imputed_mask"].sum()),
        "interpolation": int((reconstruction["imputation_method"] == 1).sum()),
        "imputeformer": int((reconstruction["imputation_method"] == 2).sum()),
        "bilstm_fair": int((reconstruction["imputation_method"] == 3).sum()),
    }
    display(summary)

In [ ]:
if reconstruction is not None and SELECTED_FILE is not None:
    missing_positions = np.flatnonzero(reconstruction["imputed_mask"].values)
    if missing_positions.size:
        center = int(missing_positions[len(missing_positions) // 2])
        radius = 72
        window = slice(
            max(0, center - radius), min(reconstruction.sizes["time"], center + radius)
        )
        time = reconstruction["time"].values[window]
        original = reconstruction["value"].values[window]
        filled = reconstruction["imputed_value"].values[window]
        mask = reconstruction["imputed_mask"].values[window].astype(bool)
        method = reconstruction["imputation_method"].values[window]
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.plot(time, filled, color="tab:orange", label="Reconstructed series")
        ax.scatter(
            time[~mask], original[~mask], s=10, color="black", label="Observed station"
        )
        ax.scatter(
            time[method == 1],
            filled[method == 1],
            s=25,
            marker="x",
            label="Interpolation",
        )
        ax.scatter(
            time[method == 2],
            filled[method == 2],
            s=25,
            marker="x",
            label="ImputeFormer",
        )
        ax.set_title(SELECTED_FILE.name)
        ax.legend()
        plt.tight_layout()
    else:
        print("The selected source file contains no reconstructed gaps.")